# 03 · Evaluate checkpoints

| Setting | Value |
|---|---|
| Accelerator | **GPU L4** (bfloat16) or **GPU T4 x2** (float16) — set `HW` |
| Internet | **ON** (pip, model download, SWE-bench Lite + cloning the 4 eval repos + fetching patched files from GitHub) |
| Attach | code (`scopesum-code`) unless `REPO_URL`; **data** from 01 (for `stats.json`/K); the **run outputs** of 02 for each variant |
| Expected runtime | long-range ~5 min/variant (200 files); localization depends on #tasks × #candidates (~1–2 s per candidate batch); probes ~5 min/variant |

Results are written per variant to `/kaggle/working/scopesum_eval/<variant>.json` and combined in `all_results.json` / `*.csv`. A variant whose JSON already exists is skipped, so you can split variants across sessions (attach the previous output and copy its JSONs in).

In [ ]:
HW = "t4"            # "t4" (float16; Kaggle GPU T4 x2) or "l4" (bfloat16)
DATA_DIR = "/kaggle/input/scopesum-data/scopesum_data"   # auto-detected below if it does not exist
VARIANTS = ["A", "B", "C", "D", "E", "F", "G"]   # evaluated in this order; variants without a checkpoint are skipped
CHECKPOINTS = {}     # variant -> checkpoint dir; {} = auto-detect .../variant_X/final in the attached inputs
RUN = {"longrange": True, "localize": True, "memory": True, "probe": True}
EVAL_OVERRIDES = {"max_eval_files": 60, "max_tasks": 20, "probe_max_files": 60}  # trimmed to fit the GPU quota; {} = full config
PREVIOUS_RESULTS = None  # e.g. "/kaggle/input/scopesum-eval-part1/scopesum_eval" to reuse finished variants


REPO_URL = "https://github.com/loaferct/scopesum.git"  # private repo: needs the Kaggle Secret GITHUB_TOKEN; "" = use an attached code dataset
CODE_DATASET_DIR = "/kaggle/input"
import glob, os, shutil, subprocess, sys
CODE_DIR = "/kaggle/working/scopesum"

def clone_repo(url, dest, secret_name="GITHUB_TOKEN"):
    """git clone; uses a GitHub token from Kaggle Secrets for private repos (never printed)."""
    auth_url = url
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(secret_name)
        auth_url = url.replace("https://", f"https://x-access-token:{token}@")
    except Exception as e:
        print(f"No Kaggle Secret {secret_name!r} ({type(e).__name__}); trying an anonymous clone")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, dest], capture_output=True, text=True)
    if r.returncode != 0:
        print("git clone failed:", r.stderr.replace(auth_url, url)[-500:])
        return False
    subprocess.run(["git", "-C", dest, "remote", "set-url", "origin", url])  # drop the token from .git/config
    print("cloned", url, subprocess.run(["git", "-C", dest, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
    return True

if not os.path.exists(os.path.join(CODE_DIR, "scopesum", "__init__.py")):
    if not (REPO_URL and clone_repo(REPO_URL, CODE_DIR)):
        hits = [h for h in glob.glob(CODE_DATASET_DIR + "/**/scopesum/__init__.py", recursive=True) if "/tests/" not in h]
        assert hits, "Attach the scopesum code dataset or set REPO_URL"
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE_DIR)
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)

In [ ]:
!pip uninstall -y -q torchao  # Kaggle ships torchao 0.10; peft>=0.21 refuses it and we do not use torchao
!pip install -q -r requirements.txt

In [ ]:
import json
from scopesum.config import load_config
from scopesum.data import load_stats, load_tokenizer, resolve_model_path, resolve_sum_token_id

def cfg_for(v):
    c = load_config("configs/base.yaml", f"configs/variant_{v}.yaml", f"configs/hw_{HW}.yaml",
                    overrides={"data": {"out_dir": DATA_DIR}, "eval": dict(EVAL_OVERRIDES)})
    if v == "B":
        c.variant.chunk_size = int(stats["chunk_size"])  # G sets its own K in variant_G.yaml
    return c

if not os.path.isdir(os.path.join(DATA_DIR, "windows_train")):
    found = [p for p in glob.glob("/kaggle/input/**/stats.json", recursive=True) if os.path.isdir(os.path.join(os.path.dirname(p), "windows_train"))]
    print("DATA_DIR has no prepared data; found:", found)
    DATA_DIR = os.path.dirname(found[0])
stats = load_stats(DATA_DIR)
K = int(stats["chunk_size"])
if not CHECKPOINTS:
    for v in VARIANTS:
        hits = sorted(glob.glob(f"/kaggle/input/**/variant_{v}/final/adapter/adapter_config.json", recursive=True))
        if hits:
            CHECKPOINTS[v] = os.path.dirname(os.path.dirname(hits[0]))
print("checkpoints:", json.dumps(CHECKPOINTS, indent=1))
cfg = cfg_for("C")
OUT = cfg.eval.out_dir
os.makedirs(OUT, exist_ok=True)
if PREVIOUS_RESULTS:
    for p in glob.glob(os.path.join(PREVIOUS_RESULTS, "*.json")):
        shutil.copy(p, OUT)
path = resolve_model_path(cfg)
tok = load_tokenizer(cfg, path)
sum_id = resolve_sum_token_id(tok, cfg)
assert sum_id == stats["sum_id"]
print("K =", K, "| sum_id =", sum_id)

## Task discovery (SWE-bench Lite)
Prints the dataset's keys, per-repo task counts and one example. Default fields: `instance_id`, `problem_statement`, `patch`, `repo`, `base_commit`. If anything differs, fix it in `FIELD_FIXES` below.

In [ ]:
from scopesum.eval_localize import discover_tasks
discover_tasks(cfg)

In [ ]:
# >>> Fix field names here after reading the discovery output (they apply to every variant) <<<
FIELD_FIXES = {
    # "issue_field": "text",
    # "patch_field": "patch",
    }
def apply_fixes(c):
    for k, v in FIELD_FIXES.items():
        setattr(c.eval, k, v)
    return c
cfg = apply_fixes(cfg)

In [ ]:
from collections import Counter
from scopesum.eval_localize import load_tasks, parse_patch, resolve_snapshot_file
from scopesum.eval_longrange import clone_eval_repos, find_eval_repo_files, prepare_long_range

# Long-range eval files: config glob, else shallow clones of fastapi / requests / rich / httpx (Internet ON)
if cfg.eval.eval_files_glob:
    eval_files = sorted(glob.glob(cfg.eval.eval_files_glob, recursive=True))[: cfg.eval.max_eval_files]
else:
    root = clone_eval_repos(cfg.eval.eval_repo_urls, cfg.eval.eval_repos_dir)
    eval_files = find_eval_repo_files(root, cfg.data.excluded_repos, cfg.eval.max_eval_files)
print(len(eval_files), "eval files;", Counter(next((r for r in cfg.data.excluded_repos if f"/{r}/" in p.lower()), "?") for p in eval_files))
lr_items = prepare_long_range(cfg, tok, eval_files)
print(len(lr_items), "windows;", sum(len(i["targets"]) for i in lr_items), "long-range target tokens")

# Localization tasks (SWE-bench Lite): check that the first task's patched files can be fetched
tasks = load_tasks(cfg)
print(len(tasks), "tasks")
t0 = tasks[0]
files0 = parse_patch(str(t0.get(cfg.eval.patch_field, "")))
print("first task:", t0.get(cfg.eval.task_id_field), "| patch files:", {f: sorted(l)[:5] for f, l in files0.items()})
print("resolved:", {f: resolve_snapshot_file(cfg, t0, f) for f in files0})

## Run all evaluations per variant

In [ ]:
import gc, time
import torch
from scopesum.data import build_example
from scopesum.eval_localize import evaluate_localization
from scopesum.eval_longrange import evaluate_long_range, save_json
from scopesum.eval_memory import analytic_report, measured_peak
from scopesum.model import describe_attention_layout, load_for_eval
from scopesum.probe import collect_features, train_probes

mem_items = [i for i in lr_items if len(i["window"]["raw_ids"]) > 1024][:16] or lr_items[:16]

for v, ck in CHECKPOINTS.items():
    out_path = os.path.join(OUT, f"{v}.json")
    if os.path.exists(out_path):
        print(f"[{v}] already done -> {out_path}"); continue
    cv = apply_fixes(cfg_for(v))
    t0 = time.time()
    lm = load_for_eval(cv, path, sum_id, ck)
    layout = describe_attention_layout(lm.model)
    res = {"variant": v, "checkpoint": ck, "layout": {k: layout[k] for k in layout if k != "layer_types"}}
    chunk = cv.variant.chunk_size if cv.variant.segment_mode == "fixed" else None
    if RUN["longrange"]:
        res["longrange"] = evaluate_long_range(lm, lr_items, tok, chunk); print(v, res["longrange"])
    if RUN["localize"]:
        res["localize"] = evaluate_localization(lm, tok, cv, tasks, chunk, log_path=os.path.join(OUT, f"localize_{v}.jsonl"))
        print(v, res["localize"])
    if RUN["memory"]:
        exs = [build_example(i["window"], lm.mode, tok.bos_token_id, sum_id, tok.pad_token_id, cv.data.seq_len, chunk_size=chunk or K, scope_span=lm.scope_span) for i in mem_items]
        res["memory_analytic"] = analytic_report({v: exs}, lm.window, cv.variant.w_small, layout)[v]
        res["memory_measured"] = measured_peak(lm, exs[0])
        print(v, {k: res["memory_analytic"][k] for k in ("live_mean", "live_max", "kv_mb_mean")}, res["memory_measured"])
    if RUN["probe"] and v != "A":
        feats = collect_features(lm, tok, eval_files, chunk, layers=tuple(cv.eval.probe_layers), max_files=cv.eval.probe_max_files, split_chunk=K)
        res["probe"] = train_probes(feats)
        for r in res["probe"]: print(v, r)
    res["minutes"] = (time.time() - t0) / 60
    save_json(res, out_path)
    del lm; gc.collect(); torch.cuda.empty_cache()

In [ ]:
from scopesum.report import collect_results, results_tables
allres = collect_results(OUT)
save_json(allres, os.path.join(OUT, "all_results.json"))
tables = results_tables(allres)
for name, df in tables.items():
    df.to_csv(os.path.join(OUT, f"{name}.csv"), index=False)
    print(f"\n== {name} =="); print(df.to_string(index=False))